# PatchCore + WideResNet50-2 Multilayer Training

This notebook runs the current best anomaly detector from the report on a larger labeled Modal split.

Model setup:
- PatchCore with a frozen pretrained `WideResNet50-2`
- multilayer feature bank from `layer2 + layer3`
- validation-threshold evaluation at the `0.95` normal quantile
- small comparison sweep around the report winner `topk_mb50k_r010`


In [ ]:
import os
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
BUNDLE_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "helpers" / "patchcore_wrn50_modal.py").exists():
        BUNDLE_ROOT = candidate
        break

if BUNDLE_ROOT is None:
    raise RuntimeError("Could not locate the WRN PatchCore Modal bundle root.")

HELPERS_ROOT = BUNDLE_ROOT / "helpers"
if str(HELPERS_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPERS_ROOT))

from patchcore_wrn50_modal import (
    DEFAULT_SPLIT_CONFIG,
    DEFAULT_VARIANTS,
    WaferArrayDataset,
    attach_scores_to_metadata,
    auto_find_raw_pickle,
    defect_type_summary,
    load_wafer_array,
    prepare_dataset,
    resolve_data_root,
    resolve_device,
    resolve_output_root,
    run_patchcore_variant,
    set_seed,
    split_summary_wide,
)


In [ ]:
RAW_PICKLE = os.environ.get("WM811K_RAW_PICKLE")
IMAGE_SIZE = 64
SEED = 42
BATCH_SIZE = 64
NUM_WORKERS = 0
DEVICE = "auto"
TEACHER_LAYERS = ["layer2", "layer3"]
PRETRAINED = True
FREEZE_BACKBONE = True
NORMALIZE_IMAGENET = True
BACKBONE_INPUT_SIZE = 224
QUERY_CHUNK_SIZE = 1024
MEMORY_CHUNK_SIZE = 4096
THRESHOLD_QUANTILE = 0.95
THRESHOLD_STRATEGY = "validation_f1"
MAX_VALIDATION_FALSE_POSITIVE_RATE = None
SPLIT_CONFIG = DEFAULT_SPLIT_CONFIG.copy()
SWEEP_VARIANTS = DEFAULT_VARIANTS.copy()
DATA_ROOT = resolve_data_root(BUNDLE_ROOT)
OUTPUT_ROOT = resolve_output_root(BUNDLE_ROOT)
ARTIFACT_DIR = OUTPUT_ROOT / "patchcore_wrn50_multilayer_120k_5pct"

set_seed(SEED)
device = resolve_device(DEVICE)
raw_pickle = auto_find_raw_pickle(RAW_PICKLE)
metadata_path = prepare_dataset(raw_pickle, DATA_ROOT, IMAGE_SIZE, SPLIT_CONFIG, seed=SEED, overwrite=False)
metadata = pd.read_csv(metadata_path)

display(split_summary_wide(metadata))
display(defect_type_summary(metadata).head(18))

print("Bundle root:", BUNDLE_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Raw pickle:", raw_pickle)
print("Metadata path:", metadata_path)
print("Artifact dir:", ARTIFACT_DIR)
print("Using device:", device)
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
train_dataset = WaferArrayDataset(metadata_path, split="train", data_root=DATA_ROOT)
val_dataset = WaferArrayDataset(metadata_path, split="val", data_root=DATA_ROOT)
test_dataset = WaferArrayDataset(metadata_path, split="test", data_root=DATA_ROOT)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

variant_results = {}
rows = []
for variant in SWEEP_VARIANTS:
    print(f"\n=== Running variant: {variant['name']} ===")
    result = run_patchcore_variant(
        variant,
        train_dataset=train_dataset,
        val_loader=val_loader,
        test_loader=test_loader,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device,
        output_dir=ARTIFACT_DIR,
        seed=SEED,
        teacher_layers=TEACHER_LAYERS,
        pretrained=PRETRAINED,
        freeze_backbone=FREEZE_BACKBONE,
        backbone_input_size=BACKBONE_INPUT_SIZE,
        normalize_imagenet=NORMALIZE_IMAGENET,
        threshold_quantile=THRESHOLD_QUANTILE,
        threshold_strategy=THRESHOLD_STRATEGY,
        max_validation_false_positive_rate=MAX_VALIDATION_FALSE_POSITIVE_RATE,
        query_chunk_size=QUERY_CHUNK_SIZE,
        memory_chunk_size=MEMORY_CHUNK_SIZE,
    )
    variant_results[variant["name"]] = result
    rows.append(result["row"])

sweep_results_df = pd.DataFrame(rows).sort_values(["f1", "auroc", "auprc"], ascending=False).reset_index(drop=True)
sweep_results_df.to_csv(ARTIFACT_DIR / "patchcore_sweep_results.csv", index=False)
selected_variant_name = str(sweep_results_df.iloc[0]["name"])
selected_result = variant_results[selected_variant_name]

bundle_summary = {
    "selected_variant": selected_variant_name,
    "split_config": SPLIT_CONFIG,
    "raw_pickle": str(raw_pickle),
    "metadata_path": str(metadata_path),
    "threshold_strategy": THRESHOLD_STRATEGY,
    "max_validation_false_positive_rate": MAX_VALIDATION_FALSE_POSITIVE_RATE,
    "variants": sweep_results_df.to_dict(orient="records"),
}
(ARTIFACT_DIR / "bundle_summary.json").write_text(json.dumps(bundle_summary, indent=2), encoding="utf-8")

display(sweep_results_df)
print("Selected variant:", selected_variant_name)


In [ ]:
metrics = selected_result["metrics"]
best_sweep = selected_result["best_sweep"]
threshold = float(selected_result["threshold"])

metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": metrics["precision"]},
        {"metric": "recall", "value": metrics["recall"]},
        {"metric": "f1", "value": metrics["f1"]},
        {"metric": "auroc", "value": metrics["auroc"]},
        {"metric": "auprc", "value": metrics["auprc"]},
        {"metric": "threshold", "value": threshold},
    ]
)

display(metrics_df)
display(
    pd.DataFrame(
        metrics["confusion_matrix"],
        index=["true_normal", "true_anomaly"],
        columns=["pred_normal", "pred_anomaly"],
    )
)
print(
    f"Best sweep threshold: {best_sweep['threshold']:.6f} | "
    f"precision={best_sweep['precision']:.4f}, recall={best_sweep['recall']:.4f}, f1={best_sweep['f1']:.4f}"
)


In [ ]:
plot_df = sweep_results_df.copy()
plot_df["label"] = plot_df["name"] + "\n" + plot_df["reduction"] + ", mb=" + plot_df["memory_bank_size"].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(plot_df["label"], plot_df["f1"], color="#277da1")
axes[0].set_title("WRN PatchCore Sweep: Validation-Threshold F1")
axes[0].set_xlabel("F1")
axes[0].invert_yaxis()

axes[1].barh(plot_df["label"], plot_df["auroc"], color="#90be6d")
axes[1].set_title("WRN PatchCore Sweep: AUROC")
axes[1].set_xlabel("AUROC")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

val_scores_df = selected_result["val_scores_df"]
test_scores_df = selected_result["test_scores_df"]
threshold_sweep_df = selected_result["threshold_sweep_df"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df["score"], bins=30, alpha=0.8, color="#4d908e")
axes[0].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[0].set_title(f"Validation Score Distribution\n{selected_variant_name}")
axes[0].legend()

axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly")
axes[1].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[1].set_title(f"Test Score Distribution\n{selected_variant_name}")
axes[1].legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
plt.axvline(threshold, color="red", linestyle="--", label=f"validation threshold = {threshold:.4f}")
plt.axvline(best_sweep["threshold"], color="green", linestyle=":", label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
plt.title(f"Threshold Sweep on Test Split\n{selected_variant_name}")
plt.xlabel("Anomaly-score threshold")
plt.ylabel("Metric value")
plt.legend()
plt.show()


## Failure Analysis

This section attaches the selected PatchCore scores to the held-out test metadata and highlights the strongest false positives and false negatives at the validation-derived threshold.


In [ ]:
test_metadata = metadata[metadata["split"] == "test"].reset_index(drop=True)
scored_test_df = attach_scores_to_metadata(test_metadata, selected_result["test_scores_df"], threshold)
scored_test_df.to_csv(ARTIFACT_DIR / f"{selected_variant_name}_test_predictions.csv", index=False)

error_summary = (
    scored_test_df.groupby(["error_type", "defect_type"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["error_type", "count"], ascending=[True, False])
)
display(error_summary.head(20))

def plot_examples(frame, title, ascending, top_n=5):
    sample = frame.sort_values("score", ascending=ascending).head(top_n)
    if sample.empty:
        print(f"No rows available for {title}.")
        return
    fig, axes = plt.subplots(1, len(sample), figsize=(3.2 * len(sample), 3.2))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        wafer = load_wafer_array(DATA_ROOT, row["array_path"])
        ax.imshow(wafer, cmap="viridis")
        ax.set_title(f"{row['defect_type']}\nscore={row['score']:.3f}")
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

false_positive_df = scored_test_df[scored_test_df["error_type"] == "false_positive"]
false_negative_df = scored_test_df[scored_test_df["error_type"] == "false_negative"]
plot_examples(false_positive_df, "Top False Positives", ascending=False)
plot_examples(false_negative_df, "Top False Negatives", ascending=True)
